# 09 - Validacion ML Pipeline v2

Valida las mejoras de la Fase 0 del pipeline ML:
- Cross-validation con XGBoost (sin data leak)
- Threshold optimization (F1 vs umbral)
- Calibracion de probabilidades (Platt scaling)
- Sensitivity analysis del labeler
- Comparacion v1 vs v2

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
from pathlib import Path

from src.data.storage import Storage
from src.models.trainer import ModelTrainer
from src.models.evaluator import ModelEvaluator
from config import MODELS_DIR, PROCESSED_DIR

storage = Storage()
print('Conexion a DB OK')
print(f'Stats: {storage.stats()}')

## 1. Cargar datos

In [ ]:
# Cargar features y labels
features_df = storage.get_features_df()
labels_df = storage.query('SELECT * FROM labels')

print(f'Features: {features_df.shape}')
print(f'Labels: {labels_df.shape}')
print(f'\nDistribucion de labels:')
print(labels_df['label_multi'].value_counts())
print(f'\nDistribucion binaria:')
print(labels_df['label_binary'].value_counts())

## 2. Entrenar con calibracion (Pipeline v2)

In [ ]:
# Entrenar modelos v2 con calibracion automatica
trainer = ModelTrainer()
results = trainer.train_all(features_df, labels_df, target='label_binary')

# Mostrar resultados
for model_name, metrics in results.get('models', {}).items():
    print(f'\n=== {model_name} ===')
    for k in ['accuracy', 'precision', 'recall', 'f1', 'roc_auc', 'pr_auc']:
        v = metrics.get(k)
        if v is not None:
            print(f'  {k}: {v:.4f}')

## 3. Threshold Optimization

Buscamos el umbral de probabilidad que maximiza F1-score.
Por defecto es 0.5, pero puede no ser optimo con datos desbalanceados.

In [ ]:
from sklearn.metrics import f1_score, precision_score, recall_score

# Obtener predicciones del mejor modelo
if hasattr(trainer, '_X_test') and hasattr(trainer, '_y_test'):
    X_test = trainer._X_test
    y_test = trainer._y_test
    
    # Usar el modelo XGBoost si existe
    model = trainer.models.get('XGBoost') or trainer.models.get('RandomForest')
    model_name = 'XGBoost' if 'XGBoost' in trainer.models else 'RandomForest'
    
    if hasattr(model, 'predict_proba'):
        probas = model.predict_proba(X_test)[:, 1]
        
        # Barrer thresholds
        thresholds = np.arange(0.1, 0.95, 0.05)
        f1_scores = []
        precisions = []
        recalls = []
        
        for t in thresholds:
            preds = (probas >= t).astype(int)
            f1_scores.append(f1_score(y_test, preds, zero_division=0))
            precisions.append(precision_score(y_test, preds, zero_division=0))
            recalls.append(recall_score(y_test, preds, zero_division=0))
        
        best_idx = np.argmax(f1_scores)
        best_threshold = thresholds[best_idx]
        best_f1 = f1_scores[best_idx]
        
        print(f'Threshold optimo: {best_threshold:.2f}')
        print(f'F1 en threshold optimo: {best_f1:.4f}')
        print(f'F1 en threshold 0.5: {f1_score(y_test, (probas >= 0.5).astype(int), zero_division=0):.4f}')
        
        # Grafico
        fig, ax = plt.subplots(figsize=(10, 6))
        ax.plot(thresholds, f1_scores, 'b-o', label='F1-Score', linewidth=2)
        ax.plot(thresholds, precisions, 'g--', label='Precision', alpha=0.7)
        ax.plot(thresholds, recalls, 'r--', label='Recall', alpha=0.7)
        ax.axvline(best_threshold, color='blue', linestyle=':', alpha=0.5,
                   label=f'Optimo: {best_threshold:.2f}')
        ax.axvline(0.5, color='gray', linestyle=':', alpha=0.5, label='Default: 0.50')
        ax.set_xlabel('Threshold')
        ax.set_ylabel('Score')
        ax.set_title(f'Threshold Optimization - {model_name}')
        ax.legend()
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.savefig(PROCESSED_DIR / 'threshold_optimization.png', dpi=150)
        plt.show()
    else:
        print('Modelo sin predict_proba, no se puede optimizar threshold')
else:
    print('No hay datos de test disponibles (entrena primero)')

## 4. Curva de Calibracion

Verifica si las probabilidades del modelo estan bien calibradas.
Una probabilidad de 0.7 deberia significar ~70% de acierto real.

In [ ]:
from sklearn.calibration import calibration_curve

if hasattr(trainer, '_X_test') and hasattr(trainer, '_y_test'):
    fig, ax = plt.subplots(figsize=(8, 8))
    
    # Linea de calibracion perfecta
    ax.plot([0, 1], [0, 1], 'k--', label='Calibracion perfecta')
    
    for name, model in trainer.models.items():
        if hasattr(model, 'predict_proba'):
            probas = model.predict_proba(X_test)[:, 1]
            
            # n_bins adaptado al tamaño del dataset
            n_bins = min(10, max(3, len(y_test) // 5))
            
            try:
                prob_true, prob_pred = calibration_curve(
                    y_test, probas, n_bins=n_bins, strategy='uniform'
                )
                ax.plot(prob_pred, prob_true, 's-', label=name, linewidth=2)
            except Exception as e:
                print(f'Error calibracion {name}: {e}')
    
    ax.set_xlabel('Probabilidad predicha')
    ax.set_ylabel('Fraccion de positivos reales')
    ax.set_title('Curva de Calibracion')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(PROCESSED_DIR / 'calibration_curve.png', dpi=150)
    plt.show()
else:
    print('No hay datos de test disponibles')

## 5. Sensitivity Analysis del Labeler

Verifica como cambian las etiquetas al variar los umbrales de clasificacion.

In [ ]:
from src.models.labeler import Labeler

labeler = Labeler(storage)

# Sensibilidad al umbral binario
binary_thresholds = [2.0, 3.0, 5.0, 7.0, 10.0]
print('Sensibilidad al umbral binario:')
print(f'{"Threshold":>10} | {"Positivos":>10} | {"Negativos":>10} | {"Ratio":>10}')
print('-' * 50)

for threshold in binary_thresholds:
    labels = labels_df.copy()
    # Simular cambio de umbral
    labels['label_binary_sim'] = (labels['max_multiple'] >= threshold).astype(int)
    pos = labels['label_binary_sim'].sum()
    neg = len(labels) - pos
    ratio = pos / max(neg, 1)
    print(f'{threshold:>10.1f} | {pos:>10d} | {neg:>10d} | {ratio:>10.2f}')

## 6. Comparacion v1 vs v2

Compara resultados del modelo original (si existe) con el re-entrenado.

In [ ]:
# Cargar resultados previos si existen
eval_path = MODELS_DIR / 'evaluation_results.json'

if eval_path.exists():
    with open(eval_path, 'r') as f:
        eval_results = json.load(f)
    
    print('Resultados del ultimo entrenamiento:')
    for model_name, metrics in eval_results.get('models', {}).items():
        print(f'\n=== {model_name} ===')
        for k in ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']:
            v = metrics.get(k)
            if v is not None:
                print(f'  {k}: {v:.4f}')
    
    # Comparar con v2
    v2_models = results.get('models', {})
    if v2_models:
        print('\n\n=== COMPARACION v1 vs v2 ===')
        for model_name in v2_models:
            v1 = eval_results.get('models', {}).get(model_name, {})
            v2 = v2_models[model_name]
            if v1:
                print(f'\n--- {model_name} ---')
                for k in ['accuracy', 'f1', 'roc_auc']:
                    v1_val = v1.get(k, 0)
                    v2_val = v2.get(k, 0)
                    diff = v2_val - v1_val if v1_val and v2_val else 0
                    arrow = '+' if diff > 0 else ''
                    print(f'  {k}: v1={v1_val:.4f} -> v2={v2_val:.4f} ({arrow}{diff:.4f})')
else:
    print('No hay resultados previos para comparar.')
    print('Ejecuta retrain.sh primero.')